## Vectorization Practice

This notebook contains several functions implemented in a naive (loop-based) way and their vectorized counterparts (`_vec`) that are currently marked with `TODO: Vectorize this`. Your task is to implement the vectorized versions of these functions using PyTorch operations, avoiding explicit Python loops wherever possible. This will significantly improve performance.

For each `_vec` function, an instruction cell will guide you on how to vectorize it.

In [ ]:
import torch

def relu_naive(x: torch.Tensor) -> torch.Tensor:
    """Sets all negative values in a 1D tensor to 0."""
    out = torch.empty_like(x)
    for i in range(len(x)):
        if x[i] < 0:
            out[i] = 0.0
        else:
            out[i] = x[i]
    return out

def relu_vec(x: torch.Tensor) -> torch.Tensor:
    return torch.relu(x)

### relu_vec

**Instruction**: For `relu_vec`, you need to implement the Rectified Linear Unit (ReLU) activation function. The `torch.relu` function provides a vectorized implementation that sets all negative values to zero while leaving positive values unchanged. Replace the `TODO` with a call to this function.

In [ ]:
import torch

def row_max_naive(matrix: torch.Tensor) -> torch.Tensor:
    """Finds the maximum value in each row of a 2D matrix (N, D)."""
    N, D = matrix.shape
    out = torch.empty(N)
    for i in range(N):
        current_max = matrix[i, 0]
        for j in range(1, D):
            if matrix[i, j] > current_max:
                current_max = matrix[i, j]
        out[i] = current_max
    return out

def row_max_vec(matrix: torch.Tensor) -> torch.Tensor:
    return torch.max(matrix, dim=1).values

### row_max_vec

**Instruction**: For `row_max_vec`, you need to find the maximum value in each row of a 2D tensor. The `torch.max` function can be used for this by specifying the `dim` parameter. Remember that `torch.max` returns both the maximum values and their indices; you only need the values here.

In [ ]:
import torch

def center_data_naive(X: torch.Tensor) -> torch.Tensor:
    N, D = X.shape
    out = torch.empty_like(X)
    col_means = torch.zeros(D)
    for j in range(D):
        for i in range(N):
            col_means[j] += X[i, j]
        col_means[j] /= N
    for i in range(N):
        for j in range(D):
            out[i, j] = X[i, j] - col_means[j]
    return out

def center_data_vec(X: torch.Tensor) -> torch.Tensor:
    return X - torch.mean(X, dim=0)

### center_data_vec

**Instruction**: For `center_data_vec`, you need to subtract the column mean from every element in that column. First, calculate the mean of each column using `torch.mean` with the appropriate `dim` parameter. Then, broadcast this mean across the rows to subtract it from the original tensor.

### get_target_probs_vec

**Instruction**: For `get_target_probs_vec`, you need to extract the probability of the true class for each item in a batch. This can be achieved using advanced indexing with `torch.arange` for the batch dimension and `labels` for the class dimension.

In [ ]:
import torch

def get_target_probs_naive(probs: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    B = probs.shape[0]
    out = torch.empty(B)
    for i in range(B):
        correct_class_idx = labels[i]
        out[i] = probs[i, correct_class_idx]
    return out

def get_target_probs_vec(probs: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return probs[torch.arange(probs.shape[0]), labels]

### one_hot_vec

**Instruction**: For `one_hot_vec`, you need to convert a 1D tensor of integer labels into a 2D one-hot encoded matrix. `torch.nn.functional.one_hot` or creating a zero tensor and using `scatter_` or `scatter_` will be useful here.

In [ ]:
import torch
import torch.nn.functional as F

def one_hot_naive(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    B = labels.shape[0]
    out = torch.zeros(B, num_classes)
    for i in range(B):
        class_idx = labels[i]
        out[i, class_idx] = 1.0
    return out

def one_hot_vec(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    B = labels.shape[0]
    out = torch.zeros(B, num_classes, device=labels.device)

    return out.scatter_(1, labels.unsqueeze(1), 1.0)

### cosine_sim_vec

**Instruction**: For `cosine_sim_vec`, you need to calculate pairwise cosine similarity. This involves dot products and L2 norms. You can use `torch.matmul` for dot products, and `torch.norm` for L2 norms. Remember to handle potential division by zero if norms are zero.

In [ ]:
import torch

def cosine_sim_naive(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    N, D = A.shape
    M = B.shape[0]
    out = torch.empty(N, M)
    for i in range(N):
        for j in range(M):
            dot_product = torch.dot(A[i], B[j])
            norm_a = torch.norm(A[i])
            norm_b = torch.norm(B[j])
            out[i, j] = dot_product / (norm_a * norm_b)
    return out

def cosine_sim_vec(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    A_norm = A / A.norm(dim=1, keepdim=True)
    B_norm = B / B.norm(dim=1, keepdim=True)
    return torch.matmul(A_norm, B_norm.t())

### class_scaling_vec

**Instruction**: For `class_scaling_vec`, you need to multiply features by a class-specific scaling factor. Use `labels` to index into `scales` to get the correct multiplier for each batch element. Then, use broadcasting to multiply these multipliers with the `features` tensor.

In [ ]:
import torch

def class_scaling_naive(features: torch.Tensor, labels: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    B, D = features.shape
    out = torch.empty_like(features)
    for i in range(B):
        class_idx = labels[i]
        multiplier = scales[class_idx]
        for j in range(D):
            out[i, j] = features[i, j] * multiplier
    return out

def class_scaling_vec(features: torch.Tensor, labels: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    return features * scales[labels].unsqueeze(1)

### causal_mask_vec

**Instruction**: For `causal_mask_vec`, you need to create a square mask where the upper triangle (elements `[i, j]` where `j > i`) is set to negative infinity. `torch.ones`, `torch.triu`, and setting elements to `float('-inf')` can be used to achieve this efficiently.

In [ ]:
import torch

def causal_mask_naive(seq_len: int) -> torch.Tensor:
    mask = torch.zeros(seq_len, seq_len)
    for i in range(seq_len):
        for j in range(seq_len):
            if j > i:
                mask[i, j] = float('-inf')
    return mask

def causal_mask_vec(seq_len: int) -> torch.Tensor:
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    return mask.masked_fill(mask == 1, float('-inf'))

### mean_pooling_vec

**Instruction**: For `mean_pooling_vec`, you need to compute the mean of variable-length sequences. You can create a mask based on `lengths`, multiply it with `seqs`, sum along the sequence dimension, and then divide by the `lengths` (after unsqueezing `lengths` to allow broadcasting).

In [ ]:
import torch

def mean_pooling_naive(seqs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    B, S, D = seqs.shape
    out = torch.zeros(B, D)
    for b in range(B):
        for i in range(lengths[b]):
            for j in range(D):
                out[b, j] += seqs[b, i, j]
        for j in range(D):
            out[b, j] /= lengths[b]
    return out

def mean_pooling_vec(seqs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = torch.arange(seqs.size(1)).unsqueeze(0) < lengths.unsqueeze(1)
    return (seqs * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(1)

### normalize_images_vec

**Instruction**: For `normalize_images_vec`, you need to normalize each image in the batch to the `[0, 1]` range independently. Find the min and max for each image using `torch.min` and `torch.max` with `dim=(1, 2)` and `keepdim=True`. Then, perform the normalization using broadcasting.

In [ ]:
import torch

def normalize_images_naive(images: torch.Tensor) -> torch.Tensor:
    B, H, W = images.shape
    out = torch.empty_like(images)
    for i in range(B):
        img_min = images[i].min()
        img_max = images[i].max()
        out[i] = (images[i] - img_min) / (img_max - img_min)
    return out

def normalize_images_vec(images: torch.Tensor) -> torch.Tensor:
    # Using the concise version you suggested:
    img_min = images.min(dim=(1, 2), keepdim=True).values
    img_max = images.max(dim=(1, 2), keepdim=True).values
    return (images - img_min) / (img_max - img_min)

### label_smoothing_vec

**Instruction**: For `label_smoothing_vec`, you need to create a smoothed one-hot encoding. Start by creating a one-hot encoding using `torch.nn.functional.one_hot`. Then, adjust the values according to the label smoothing formula, where incorrect classes get `epsilon / (num_classes - 1)` and the correct class gets `1 - epsilon`.

In [ ]:
import torch
import torch.nn.functional as F

def label_smoothing_naive(labels: torch.Tensor, num_classes: int, epsilon: float) -> torch.Tensor:
    B = labels.shape[0]
    out = torch.empty(B, num_classes)
    incorrect_prob = epsilon / (num_classes - 1)
    correct_prob = 1.0 - epsilon
    for i in range(B):
        true_class = labels[i]
        for c in range(num_classes):
            out[i, c] = correct_prob if c == true_class else incorrect_prob
    return out

def label_smoothing_vec(labels: torch.Tensor, num_classes: int, epsilon: float) -> torch.Tensor:
    soft_labels = torch.full((labels.size(0), num_classes), epsilon / (num_classes - 1))
    soft_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    return soft_labels

### top_k_accuracy_vec

**Instruction**: For `top_k_accuracy_vec`, you need to check if the true label is among the top `k` predictions for each item in a batch. Use `torch.topk` to get the top `k` indices from `logits`. Then, compare these top `k` indices with the `labels` using broadcasting and `torch.any`.

In [ ]:
import torch

def top_k_accuracy_naive(logits: torch.Tensor, labels: torch.Tensor, k: int) -> torch.Tensor:
    B, C = logits.shape
    out = torch.zeros(B, dtype=torch.bool)
    for i in range(B):
        top_k_indices = torch.topk(logits[i], k).indices
        if (top_k_indices == labels[i]).any():
            out[i] = True
    return out

def top_k_accuracy_vec(logits: torch.Tensor, labels: torch.Tensor, k: int) -> torch.Tensor:
    top_k_indices = torch.topk(logits, k, dim=1).indices
    return (top_k_indices == labels.unsqueeze(1)).any(dim=1)

### get_last_token_vec

**Instruction**: For `get_last_token_vec`, you need to extract the hidden state of the last valid token for each sequence. This can be done using advanced indexing. Create a `torch.arange` for the batch dimension and use `lengths - 1` for the sequence dimension.

In [ ]:
import torch

def get_last_token_naive(hidden_states: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    B, S, D = hidden_states.shape
    out = torch.empty(B, D)
    for i in range(B):
        last_idx = lengths[i] - 1
        out[i] = hidden_states[i, last_idx]
    return out

def get_last_token_vec(hidden_states: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    return hidden_states[torch.arange(hidden_states.size(0)), lengths - 1]

### box_iou_vec

**Instruction**: For `box_iou_vec`, you need to calculate pairwise Intersection over Union (IoU) between two sets of bounding boxes. This involves calculating intersection coordinates and areas. Use broadcasting (`unsqueeze` to expand dimensions for `boxes1` and `boxes2`) and element-wise operations (`max`, `min`) to compute intersection and union areas without explicit loops.

In [ ]:
import torch

def box_iou_naive(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """
    Args:
        boxes1: Tensor of shape (N, 4)
        boxes2: Tensor of shape (M, 4)
    Returns:
        iou: Tensor of shape (N, M) containing the IoU for every pair.
    """
    N = boxes1.shape[0]
    M = boxes2.shape[0]
    iou = torch.empty(N, M)
    for i in range(N):
        for j in range(M):
            b1 = boxes1[i]
            b2 = boxes2[j]
            # Intersection rectangle
            inter_x1 = max(b1[0], b2[0])
            inter_y1 = max(b1[1], b2[1])
            inter_x2 = min(b1[2], b2[2])
            inter_y2 = min(b1[3], b2[3])
            inter_area = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)
            # Areas
            area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
            area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
            iou[i, j] = inter_area / (area1 + area2 - inter_area)
    return iou

def box_iou_vec(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    b1 = boxes1.unsqueeze(1)
    b2 = boxes2.unsqueeze(0)
    inter_x1 = torch.max(b1[..., 0], b2[..., 0])
    inter_y1 = torch.max(b1[..., 1], b2[..., 1])
    inter_x2 = torch.min(b1[..., 2], b2[..., 2])
    inter_y2 = torch.min(b1[..., 3], b2[..., 3])
    inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    return inter_area / (area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area)

### bilinear_vec

**Instruction**: For `bilinear_vec`, you need to implement the bilinear product. This operation can be expressed using `torch.einsum` or a combination of `torch.matmul` and `permute`/`transpose` operations to efficiently compute the sum `x1[b, i] * W[o, i, j] * x2[b, j]`.

In [ ]:
import torch

def bilinear_naive(x1: torch.Tensor, x2: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    B, D1 = x1.shape
    out_dim, _, D2 = W.shape
    out = torch.zeros(B, out_dim)
    for b in range(B):
        for o in range(out_dim):
            for i in range(D1):
                for j in range(D2):
                    out[b, o] += x1[b, i] * W[o, i, j] * x2[b, j]
    return out

def bilinear_vec(x1: torch.Tensor, x2: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    return torch.einsum('bi,oij,bj->bo', x1, W, x2)

### aggregate_nodes_vec

**Instruction**: For `aggregate_nodes_vec`, you need to sum features based on `group_indices`. `torch.zeros` to initialize the output and then `scatter_add_` is a very efficient way to perform this aggregation in a vectorized manner.

In [ ]:
import torch

def aggregate_nodes_naive(features: torch.Tensor, group_indices: torch.Tensor, num_groups: int) -> torch.Tensor:
    N, D = features.shape
    out = torch.zeros(num_groups, D)
    for i in range(N):
        group_id = group_indices[i]
        for j in range(D):
            out[group_id, j] += features[i, j]
    return out

def aggregate_nodes_vec(features: torch.Tensor, group_indices: torch.Tensor, num_groups: int) -> torch.Tensor:
    out = torch.zeros(num_groups, features.size(1), device=features.device)
    return out.scatter_add_(0, group_indices.unsqueeze(1).expand(-1, features.size(1)), features)

### split_heads_vec

**Instruction**: For `split_heads_vec`, you need to reshape the `qkv` tensor from `(B, S, D)` to `(B, num_heads, S, head_dim)`. This can be done using `view` and `permute` operations to achieve the desired output shape.

In [ ]:
import torch

def split_heads_naive(qkv: torch.Tensor, num_heads: int) -> torch.Tensor:
    B, S, D = qkv.shape
    h_dim = D // num_heads
    out = torch.empty(B, num_heads, S, h_dim)
    for b in range(B):
        for s in range(S):
            for h in range(num_heads):
                for d in range(h_dim):
                    out[b, h, s, d] = qkv[b, s, h * h_dim + d]
    return out

def split_heads_vec(qkv: torch.Tensor, num_heads: int) -> torch.Tensor:
    B, S, D = qkv.shape
    return qkv.view(B, S, num_heads, D // num_heads).permute(0, 2, 1, 3)

### rolling_window_vec

**Instruction**: For `rolling_window_vec`, you need to compute the sum of elements within a rolling window. `torch.nn.functional.unfold` or creating shifted views of the `series` tensor and then summing them can achieve this efficiently.

In [ ]:
import torch
import torch.nn.functional as F

def rolling_window_naive(series: torch.Tensor, window_size: int) -> torch.Tensor:
    L = series.shape[0]
    num_w = L - window_size + 1
    out = torch.zeros(num_w)
    for i in range(num_w):
        for j in range(window_size):
            out[i] += series[i+j]
    return out

def rolling_window_vec(series: torch.Tensor, window_size: int) -> torch.Tensor:
    return series.unfold(0, window_size, 1).sum(dim=1)

### vit_patching_vec

**Instruction**: For `vit_patching_vec`, you need to convert an image tensor into a sequence of flattened patches. This can be achieved by using `torch.nn.functional.unfold` which reshapes and extracts sliding local blocks, and then `permute` and `flatten` or `reshape` to get the final `(B, Num_Patches, C * P * P)` shape.

In [ ]:
import torch
import torch.nn.functional as F

def vit_patching_naive(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    B, C, H, W = images.shape
    P = patch_size
    num_p = (H // P) * (W // P)
    out = torch.empty(B, num_p, C * P * P)
    for b in range(B):
        idx = 0
        for y in range(0, H, P):
            for x in range(0, W, P):
                flat_val = images[b, :, y:y+P, x:x+P].flatten()
                out[b, idx] = flat_val
                idx += 1
    return out

def vit_patching_vec(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    patches = F.unfold(images, kernel_size=patch_size, stride=patch_size)
    return patches.transpose(1, 2)

### seq_loss_vec

**Instruction**: For `seq_loss_vec`, you need to calculate the average negative log-likelihood, ignoring padded tokens (`-100`). This is a common task in sequence models. Use `torch.nn.functional.nll_loss` (Negative Log Likelihood Loss) and specify `ignore_index=-100` to handle the padded tokens. Ensure your `log_probs` are in the correct format for this function (typically `(N, C, ...)`, where `C` is `V` in your case, so you might need to permute dimensions).

In [ ]:
import torch
import torch.nn.functional as F

def seq_loss_naive(log_probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    B, S, V = log_probs.shape
    total_loss, count = 0.0, 0
    for b in range(B):
        for s in range(S):
            if targets[b, s] != -100:
                total_loss -= log_probs[b, s, targets[b, s]]
                count += 1
    return total_loss / count

def seq_loss_vec(log_probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    B, S, V = log_probs.shape

    batch_idx = torch.arange(B).view(B, 1).expand(B, S)
    seq_idx = torch.arange(S).view(1, S).expand(B, S)
    target_log_probs = log_probs[batch_idx, seq_idx, targets.clamp(min=0)]
    mask = (targets != -100).float()
    return -(target_log_probs * mask).sum() / mask.sum()

### Alternative: Sequence Loss via Flattening
If advanced indexing feels complex, you can flatten the batch and sequence dimensions into one large dimension. This turns the 3D problem into a 2D problem where you only need a 1D index to select the correct class for each position.

In [ ]:
def seq_loss_flattened(log_probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    B, S, V = log_probs.shape

    logits_flat = log_probs.view(-1, V)
    targets_flat = targets.view(-1)

    row_idx = torch.arange(logits_flat.size(0))
    target_log_probs = logits_flat[row_idx, targets_flat.clamp(min=0)]

    mask = (targets_flat != -100).float()
    return -(target_log_probs * mask).sum() / mask.sum()

### contrastive_matrix_vec

**Instruction**: For `contrastive_matrix_vec`, you need to compute pairwise cosine similarities between embeddings in a batch, setting the diagonal to negative infinity. You can calculate the dot products using `torch.matmul` between `embeddings` and its transpose. Then, compute the L2 norms for each embedding, and divide the dot products by the outer product of these norms. Finally, use `torch.fill_diagonal_` to set the diagonal elements to `-inf`.

In [ ]:
import torch

def contrastive_matrix_naive(embeddings: torch.Tensor) -> torch.Tensor:
    B, D = embeddings.shape
    out = torch.empty(B, B)
    for i in range(B):
        for j in range(B):
            if i == j:
                out[i, j] = float('-inf')
            else:
                out[i, j] = torch.dot(embeddings[i], embeddings[j]) / (torch.norm(embeddings[i]) * torch.norm(embeddings[j]))
    return out

def contrastive_matrix_vec(embeddings: torch.Tensor) -> torch.Tensor:
    norm = embeddings.norm(dim=1, keepdim=True)
    sim = torch.matmul(embeddings, embeddings.t()) / torch.matmul(norm, norm.t())
    return sim.fill_diagonal_(float('-inf'))

## Additional Challenges

### topk_masking_vec

**Instruction**: Use `torch.topk` to get the indices of the largest `k` elements along the last dimension. Then, use `scatter_` to fill a zero-tensor with ones at those specific indices.

In [ ]:
import torch

def topk_masking_naive(scores: torch.Tensor, k: int) -> torch.Tensor:
    """
    Creates a mask where only the top-k values in each row are 1, and others are 0.
    Args:
        scores: Tensor of shape (B, D)
        k: Number of top elements to keep
    """
    B, D = scores.shape
    mask = torch.zeros_like(scores)
    for i in range(B):
        row = scores[i]
        indices = torch.argsort(row, descending=True)[:k]
        for idx in indices:
            mask[i, idx] = 1.0
    return mask

def topk_masking_vec(scores: torch.Tensor, k: int) -> torch.Tensor:
    indices = torch.topk(scores, k, dim=1).indices
    return torch.zeros_like(scores).scatter_(1, indices, 1.0)

### batch_matmul_scaled_vec

**Instruction**: Use `torch.bmm` (Batch Matrix Multiplication) to compute the product for all batches at once. Then, use broadcasting to multiply the resulting `(B, N, P)` tensor by the `scales` tensor of shape `(B,)`.

In [ ]:
import torch

def batch_matmul_scaled_naive(A: torch.Tensor, B: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    """
    Performs batch matrix multiplication and scales each batch matrix individually.
    Args:
        A: Tensor of shape (B, N, M)
        B: Tensor of shape (B, M, P)
        scales: Tensor of shape (B,) scaling factor for each batch
    """
    Batch, N, M = A.shape
    P = B.shape[2]
    out = torch.zeros(Batch, N, P)

    for b in range(Batch):
        for i in range(N):
            for j in range(P):
                for k_inner in range(M):
                    out[b, i, j] += A[b, i, k_inner] * B[b, k_inner, j]

        for i in range(N):
            for j in range(P):
                out[b, i, j] *= scales[b]
    return out

def batch_matmul_scaled_vec(A: torch.Tensor, B: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    return torch.bmm(A, B) * scales.view(-1, 1, 1)

### cumulative_sum_vec

**Instruction**: PyTorch provides a dedicated built-in function for this operation. Use `torch.cumsum` on the input tensor.

In [ ]:
import torch

def cumulative_sum_naive(x: torch.Tensor) -> torch.Tensor:
    """
    Computes the prefix sum of a 1D tensor.
    Example: [1, 2, 3] -> [1, 3, 6]
    """
    L = x.shape[0]
    out = torch.zeros_like(x)
    current_sum = 0.0
    for i in range(L):
        current_sum += x[i]
        out[i] = current_sum
    return out

def cumulative_sum_vec(x: torch.Tensor) -> torch.Tensor:
    return torch.cumsum(x, dim=0)